In [9]:
import re
from dataclasses import dataclass, field

def strip_comments(text):
    return re.sub(r"//.*", "", text)

def split_statements(text):
    statements, depth, current = [], 0, ""
    for ch in text:
        current += ch
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                if current.strip():
                    statements.append(current.strip())
                current = ""
        elif ch == ";" and depth == 0:
            if current.strip():
                statements.append(current.strip())
            current = ""
    if current.strip():
        statements.append(current.strip())
    return statements

@dataclass
class GateDef:
    params: list
    body: list

def parse_registers_and_gatedefs(statements):
    registers, gate_defs, remaining = {}, {}, []
    for s in statements:
        m = re.match(r"^(qreg|creg)\s+(\w+)\[(\d+)\]\s*;?$", s)
        if m:
            registers[m.group(2)] = int(m.group(3))
            continue
        m = re.match(r"^gate\s+(\w+)\s+([\w,\s]+?)\s*\{(.+)\}\s*;?$", s, re.DOTALL)
        if m:
            params = [p.strip() for p in m.group(2).split(",")]
            gate_defs[m.group(1)] = GateDef(params, split_statements(m.group(3)))
            continue
        if s.startswith("OPENQASM") or s.startswith("include"):
            continue
        remaining.append(s)
    return registers, gate_defs, remaining

@dataclass
class Op:
    kind: str
    qubits: list = field(default_factory=list)
    target_bit: str = None
    cond_register: str = None

def flatten_one(s, gate_defs, registers):
    m = re.match(r"^if\s*\(\s*(\w+)\s*==\s*(\d+)\s*\)\s*(\w+)\s+([\w\[\]]+)\s*;?$", s)
    if m:
        return [Op("if", cond_register=m.group(1))]

    m = re.match(r"^measure\s+([\w\[\]]+)\s*->\s*([\w\[\]]+)\s*;?$", s)
    if m:
        qubit_arg, bit_arg = m.group(1), m.group(2)
        if "[" not in qubit_arg and qubit_arg in registers:
            size = registers[qubit_arg]
            return [Op("measure", [f"{qubit_arg}[{i}]"], target_bit=f"{bit_arg}[{i}]") for i in range(size)]
        return [Op("measure", [qubit_arg], target_bit=bit_arg)]

    if s.startswith("barrier"):
        return [Op("barrier")]

    m = re.match(r"^(\w+)\s+([\w\[\],\s]+)\s*;?$", s)
    if m:
        gate_name = m.group(1)
        args = [a.strip() for a in m.group(2).split(",")]
        if gate_name in gate_defs:
            gdef = gate_defs[gate_name]
            sub = dict(zip(gdef.params, args))
            expanded = []
            for inner in gdef.body:
                for param, real in sub.items():
                    inner = re.sub(rf"\b{re.escape(param)}\b", real, inner)
                expanded.extend(flatten_one(inner, gate_defs, registers))
            return expanded
        return [Op(gate_name, args)]
    return []

def flatten(statements, gate_defs, registers):
    ops = []
    for s in statements:
        ops.extend(flatten_one(s, gate_defs, registers))
    return ops

def detect_stabilizer_checks(ops):
    checks, check_num = [], 0
    syndrome_registers = {op.cond_register for op in ops if op.kind == "if"}
    pending = {}
    for op in ops:
        if op.kind == "cx" and len(op.qubits) == 2:
            src, tgt = op.qubits
            pending.setdefault(tgt, []).append(src)
        elif op.kind == "measure":
            q, bit = op.qubits[0], op.target_bit
            reg = re.match(r"(\w+)", bit).group(1)
            if q in pending and reg in syndrome_registers:
                checks.append({"check_id": f"check_{check_num}", "ancilla_qubit": q,
                                "data_qubits": pending[q], "syndrome_bit": bit})
                check_num += 1
            pending.pop(q, None)
    return checks

def extract_stabilizer_checks(qasm_text):
    text = strip_comments(qasm_text)
    statements = split_statements(text)
    registers, gate_defs, remaining = parse_registers_and_gatedefs(statements)
    ops = flatten(remaining, gate_defs, registers)
    return detect_stabilizer_checks(ops)

print("Parser loaded.")

Parser loaded.


In [10]:
import io, zipfile
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client

settings = Settings.from_environment()
client = minio_client(settings)

key = "bronze/source=qasmbench/qasmbench-qec.zip"
response = client.get_object(settings.s3_bucket, key)
zip_bytes = response.read()
response.close()
response.release_conn()

circuits = [
    ("error_correctiond3_n5", "source", "small/error_correctiond3_n5/error_correctiond3_n5.qasm"),
    ("error_correctiond3_n5", "transpiled", "small/error_correctiond3_n5/error_correctiond3_n5_transpiled.qasm"),
    ("qec_en_n5", "source", "small/qec_en_n5/qec_en_n5.qasm"),
    ("qec_en_n5", "transpiled", "small/qec_en_n5/qec_en_n5_transpiled.qasm"),
    ("qec_sm_n5", "source", "small/qec_sm_n5/qec_sm_n5.qasm"),
    ("qec_sm_n5", "transpiled", "small/qec_sm_n5/qec_sm_n5_transpiled.qasm"),
]

stabilizer_check_rows = []

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for benchmark, variant, member in circuits:
        circuit_id = f"{benchmark}:{variant}"
        qasm_text = z.open(member).read().decode()
        checks = extract_stabilizer_checks(qasm_text)
        print(f"{circuit_id:35s} checks={len(checks)}")

        for check in checks:
            stabilizer_check_rows.append({
                "source_record_id": f"qasmbench:{member}:{check['check_id']}",
                "circuit_id": circuit_id,
                **check,
            })

error_correctiond3_n5:source        checks=0
error_correctiond3_n5:transpiled    checks=0
qec_en_n5:source                    checks=0
qec_en_n5:transpiled                checks=0
qec_sm_n5:source                    checks=2
qec_sm_n5:transpiled                checks=2


In [11]:
import pandas as pd

stabilizer_check_df = pd.DataFrame(stabilizer_check_rows)
display(stabilizer_check_df)

,source_record_id,circuit_id,check_id,ancilla_qubit,data_qubits,syndrome_bit
0,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm:check_0,qec_sm_n5:source,check_0,a[0],"[q[0], q[1]]",syn[0]
1,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm:check_1,qec_sm_n5:source,check_1,a[1],"[q[1], q[2]]",syn[1]
2,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled...,qec_sm_n5:transpiled,check_0,a[0],"[q[0], q[1]]",syn[0]
3,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled...,qec_sm_n5:transpiled,check_1,a[1],"[q[1], q[2]]",syn[1]
